# UniProt human proteome — ProtT5 embeddings (20 k × 1024)

[UniProt](https://www.uniprot.org/) is the central resource for protein sequence and
functional information.  This notebook maps the human reference proteome
(**UP000005640**, Homo sapiens) using per-protein
[ProtT5-XL-U50](https://github.com/agemagician/ProtTrans) embeddings precomputed by
the UniProt team and hosted on the UniProt FTP server.

**Licence:** UniProt data is licensed under
[CC BY 4.0](https://creativecommons.org/licenses/by/4.0/); verify at
<https://www.uniprot.org/help/license>.  The ProtTrans models are MIT-licensed.

**Data sources:**
- Embeddings: `per-protein.h5` from
  <https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/embeddings/UP000005640_9606/>
  (~84 MB, 20,588 proteins × 1024 dims, float16 HDF5).
- Metadata: streamed as TSV from the UniProt REST API
  (`https://rest.uniprot.org/uniprotkb/stream?…`).

**First run:** downloading the HDF5 file takes about 1 minute; reading embeddings and
fetching metadata adds another minute.  Results are cached in
`~/.cache/positron-atlas/datasets/uniprot/`.

**Quick try:** set `SAMPLE = True` to load only the first 2,000 proteins (seconds).

**Anchor budget note:** With 1024 feature dimensions, Atlas's WebGPU UMAP can use up to
10 M / 1024 ≈ 9,765 anchors.  Since the human proteome has ~20 k proteins, only about
half will be exact anchors and the rest will be placed.  For a fully exact run, enable
**Apply PCA first** (50 components) in the Atlas **Data** tab before clicking **Apply**.

**What to try in Atlas:**
- Hover a protein to see its name, gene, and family, and open it on UniProt.
- Color by `family` to see broad structural/functional groups.
- Color by `length` (protein length) with the histogram filter.
- Search for *EGFR*, *TP53*, or *BRCA1* to locate known cancer proteins.
- Enable **Apply PCA first** (50 dims) in the Data tab so UMAP runs on all 20 k proteins
  without sampling.
- Turn on cluster labels to discover functional clusters.

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
# Set SAMPLE = True to load only the first 2,000 proteins (seconds).
# Set SAMPLE = False for the full human reference proteome (~20 k proteins).
SAMPLE = False

PROTEOME      = "UP000005640_9606"   # human reference proteome
EMBEDDINGS_URL = (
    "https://ftp.uniprot.org/pub/databases/uniprot/current_release"
    "/knowledgebase/embeddings/UP000005640_9606/per-protein.h5"
)
META_URL = (
    "https://rest.uniprot.org/uniprotkb/stream"
    "?query=proteome:UP000005640&format=tsv"
    "&fields=accession,protein_name,gene_names,organism_name,protein_families,length"
)
SAMPLE_N = 2_000

In [ ]:
import time, urllib.request
from pathlib import Path

import h5py
import numpy as np
import polars as pl

CACHE = Path.home() / ".cache" / "positron-atlas" / "datasets" / "uniprot"
EMBEDDINGS_PATH = CACHE / "per-protein-UP000005640_9606.h5"
META_PATH       = CACHE / "uniprot_human_meta.tsv"
UNIPROT_PATH    = CACHE / "uniprot.parquet"
CACHE.mkdir(parents=True, exist_ok=True)


def _download(url: str, path: Path) -> Path:
    """Download url to path atomically; skip if already present."""
    if path.exists():
        return path
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".part")
    print(f"Downloading {url} ...")
    urllib.request.urlretrieve(url, tmp)
    tmp.rename(path)
    return path

In [ ]:
# ── Phase 1: download the ProtT5 HDF5 embeddings file ─────────────────────────
if UNIPROT_PATH.exists():
    print("uniprot.parquet already cached — skipping all phases")
elif EMBEDDINGS_PATH.exists():
    print(f"Embeddings already downloaded ({EMBEDDINGS_PATH.stat().st_size / 1e6:.0f} MB)")
else:
    _t0 = time.perf_counter()
    _download(EMBEDDINGS_URL, EMBEDDINGS_PATH)
    print(f"Downloaded in {time.perf_counter() - _t0:.0f} s  ({EMBEDDINGS_PATH.stat().st_size / 1e6:.0f} MB)")

In [ ]:
# ── Phase 2: stream metadata from the UniProt REST API ────────────────────────
# Fields: accession, protein name, gene names, organism, protein families, length.
# The stream returns a TSV with one row per protein; cached as a local file.

if UNIPROT_PATH.exists():
    print("uniprot.parquet already cached — skipping Phase 2")
elif META_PATH.exists():
    print(f"Metadata already cached ({META_PATH.stat().st_size / 1e3:.0f} kB)")
else:
    print("Phase 2: fetching metadata from UniProt REST API...")
    _t0 = time.perf_counter()
    _req = urllib.request.Request(META_URL, headers={"Accept": "text/plain"})
    with urllib.request.urlopen(_req, timeout=120) as _resp:
        _raw = _resp.read()
    META_PATH.write_bytes(_raw)
    print(f"Metadata saved ({META_PATH.stat().st_size / 1e3:.0f} kB) in {time.perf_counter() - _t0:.0f} s")

In [ ]:
# ── Phase 3: load embeddings + metadata, produce final DataFrame ──────────────
if UNIPROT_PATH.exists():
    print("uniprot.parquet already cached — loading")
else:
    print("Phase 3: loading embeddings and joining metadata...")
    _t0 = time.perf_counter()

    # Load embeddings from HDF5; each key is a UniProt accession.
    with h5py.File(EMBEDDINGS_PATH, "r") as _f:
        _accs = sorted(_f.keys())                             # sorted for reproducibility
        if SAMPLE:
            _accs = _accs[:SAMPLE_N]
        _embeddings = np.stack([_f[a][:] for a in _accs]).astype(np.float32)

    print(f"  Loaded {len(_accs):,} embeddings × {_embeddings.shape[1]} dims in {time.perf_counter() - _t0:.0f} s")

    # Read metadata TSV; rename columns to short names.
    _meta = (
        pl.read_csv(META_PATH, separator="	", infer_schema_length=10_000)
        .rename({
            "Entry":            "accession",
            "Protein names":    "title",
            "Gene Names":       "gene",
            "Organism":         "organism",
            "Protein families": "family",
            "Length":           "length",
        })
        .with_columns(
            # Keep only the first (recommended) protein name: text before " (".
            pl.col("title").str.replace(r"\s*\(.*", "").alias("title"),
            # Keep only the first gene name.
            pl.col("gene").str.split(" ").list.first().alias("gene"),
            # Shorten organism name: "Homo sapiens (Human)" → "Homo sapiens".
            pl.col("organism").str.replace(r"\s*\(.*", "").alias("organism"),
            # Keep only the first protein family.
            pl.col("family").str.split(",").list.first().alias("family"),
            pl.col("length").cast(pl.Int32),
        )
    )

    # Build embedding columns (e0 … e1023).
    _n_dims = _embeddings.shape[1]
    _emb_df = pl.from_numpy(_embeddings, schema=[f"e{i}" for i in range(_n_dims)])

    # Assemble final DataFrame: metadata columns + URL + embedding columns.
    _ids_df = pl.DataFrame({"accession": _accs})
    uniprot = (
        _ids_df
        .join(_meta, on="accession", how="left")
        .with_columns(
            pl.format("https://www.uniprot.org/uniprotkb/{}/entry", pl.col("accession")).alias("url"),
        )
        .select(["accession", "title", "gene", "family", "organism", "length", "url"])
        .hstack(_emb_df)
    )

    uniprot.write_parquet(UNIPROT_PATH)
    print(f"Saved uniprot.parquet ({UNIPROT_PATH.stat().st_size / 1e6:.0f} MB, {len(uniprot):,} rows) in {time.perf_counter() - _t0:.0f} s")

uniprot = pl.read_parquet(UNIPROT_PATH)
uniprot.shape

## Open it in Atlas

1. In the **Variables** pane, double-click **`uniprot`** to open it in the Data Explorer.
2. Click **Open in Atlas**, the first button in the Data Explorer's toolbar (top left).

Or run **Atlas: Open DataFrame…** from the Command Palette and pick `uniprot`.

## Things to try

- **Color by `family`.** Protein families (kinases, GPCRs, immunoglobulins, …) form
  distinct clusters.  The legend lets you toggle families on and off.
- **Color by `length`** and use the histogram slider to isolate short peptides or
  large structural proteins.
- **Search** for *EGFR*, *TP53*, *BRCA1*, or *ACTB* to find well-known proteins and
  see their neighbours.
- **Enable *Apply PCA first*** (50 components) in the Atlas **Data** tab before
  clicking **Apply**. This reduces the feature space from 1024 to 50 dims so UMAP
  runs on all 20 k proteins exactly (no anchor placement needed).
- **Hover a point** and click **Open link** to view the UniProt entry.
- **Add umap_x / umap_y** in the **Export** tab, then run:

  ```python
  uniprot.group_by("family").agg(pl.col("umap_x", "umap_y").mean()).sort("family")
  ```